# GNN tuned-GFP runs (four operators, configurations 4 and 5)

Runs on a Kaggle GPU notebook: the four operators on configurations 4 and 5, the fixed hyperparameters (`conf/hyperparameters/gnn/fixed.yaml`), with the tuned GFP sheet (64 columns) in place of the default one (61 columns). Eight runs, about 8 hours.
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the datasets `sandrokhizanishvili/hi-small-gnn` and `sandrokhizanishvili/hi-small-gfp-tuned`.
- Run All (or Save Version -> Save & Run All to run headless, 12 h limit).
- Each run writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt`, `config.yaml` to `/kaggle/working/outputs/train/<model>/<run>_gfp-tuned/seed-42/`.

In [ ]:
MODELS = 'gin,pna,gat,transformer'
FEATURE_CONFIGS = 'c4,c5'
SEED = 42

In [ ]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

In [ ]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml

In [ ]:
# One data folder: the graphs and gfp_variants/ linked from the two datasets (in /tmp, so the links stay out of the Output tab).
INPUT = '/kaggle/input/datasets/sandrokhizanishvili'
!mkdir -p /tmp/data
!ln -sf {INPUT}/hi-small-gnn/train_graph.pt {INPUT}/hi-small-gnn/val_graph.pt {INPUT}/hi-small-gnn/test_graph.pt {INPUT}/hi-small-gnn/feature_meta.json /tmp/data/
!ln -sfn {INPUT}/hi-small-gfp-tuned /tmp/data/gfp_variants
!ls -lL /tmp/data /tmp/data/gfp_variants

In [ ]:
# Environment test: versions, the GPU, one forward / backward pass per operator.
!python run.py mode=check machine=kaggle

In [ ]:
# The eight runs, one after another. A finished run is skipped, so the cell can be rerun after an interruption.
!python run.py -m model={MODELS} feature_config={FEATURE_CONFIGS} gfp_sheet=tuned seed={SEED} machine=kaggle paths.data=/tmp/data

In [ ]:
# The metrics of every finished run, one row each.
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob('/kaggle/working/outputs/train/*/*_gfp-tuned/seed-*/results.json'))]
cols = ['run', 'best_epoch', 'threshold', 'params', 'invariant_params'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished run yet"

In [ ]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

## After the run
- Output tab: download `results_small.zip` (metrics, curves, histories) or the whole `outputs` folder (adds `predictions.csv` and `best.pt`).
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the row in `EXPERIMENTS.md`.